# ModelScope: LLM Inference Optimization and Quality Evaluation Suite
## Benchmarking Llama-3.2-3B vs Gemma-2-2B Across Quantization Configs on CUDA GPU

Production ML teams must choose between model quality and inference cost.
This notebook benchmarks 8 quantization variants across speed, memory, and
three quality metrics to surface Pareto-optimal deployment configurations automatically.

Stack: PyTorch, HuggingFace Transformers, bitsandbytes, CUDA T4, Prometheus-compatible metrics

In [ ]:
# CELL 1: Verify T4 GPU, mount Drive, clone repo, install dependencies
!nvidia-smi
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/ANUVIK2401/modelscope /content/modelscope 2>/dev/null || echo 'Repo already cloned'
%cd /content/modelscope
!pip install -q transformers bitsandbytes accelerate sentence-transformers \
  prometheus-client rouge-score datasets tqdm matplotlib pandas fastapi uvicorn numpy


## Section 1: Model Access
Verifying gated model access for both architectures before loading weights.

In [ ]:
# CELL 2: Verify HuggingFace token and model access for Llama and Gemma
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer
login(token=userdata.get('HF_TOKEN'))
print('Testing Llama-3.2-3B access...')
tok = AutoTokenizer.from_pretrained('meta-llama/Llama-3.2-3B-Instruct')
print('Llama OK')
print('Testing Gemma-2-2B access...')
tok = AutoTokenizer.from_pretrained('google/gemma-2-2b-it')
print('Gemma OK')
del tok
import gc; gc.collect()


## Section 2: Inference Benchmarking
Measuring throughput (tokens/sec), latency (TTFT p50/p95), and peak GPU memory across 8 variants to quantify the speed-memory tradeoff of each quantization config.

In [ ]:
# CELL 3: Write benchmark runner to disk
import os
os.makedirs('/content/modelscope/results', exist_ok=True)
with open('/content/modelscope/benchmarks/runner.py', 'w') as f:
    f.write('import os, gc, csv, time, torch, numpy as np\n')
    f.write('from tqdm import tqdm\n')
    f.write('from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n')
    f.write('os.makedirs("/content/modelscope/results", exist_ok=True)\n')
    f.write('VARIANTS = [\n')
    f.write('    {"variant": "llama-fp16", "model_id": "meta-llama/Llama-3.2-3B-Instruct", "quant_config": None},\n')
    f.write('    {"variant": "llama-int8", "model_id": "meta-llama/Llama-3.2-3B-Instruct", "quant_config": BitsAndBytesConfig(load_in_8bit=True)},\n')
    f.write('    {"variant": "llama-int4", "model_id": "meta-llama/Llama-3.2-3B-Instruct", "quant_config": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)},\n')
    f.write('    {"variant": "llama-int4-nf4", "model_id": "meta-llama/Llama-3.2-3B-Instruct", "quant_config": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)},\n')
    f.write('    {"variant": "gemma2-fp16", "model_id": "google/gemma-2-2b-it", "quant_config": None},\n')
    f.write('    {"variant": "gemma2-int8", "model_id": "google/gemma-2-2b-it", "quant_config": BitsAndBytesConfig(load_in_8bit=True)},\n')
    f.write('    {"variant": "gemma2-int4", "model_id": "google/gemma-2-2b-it", "quant_config": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)},\n')
    f.write('    {"variant": "gemma2-int4-nf4", "model_id": "google/gemma-2-2b-it", "quant_config": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)},\n')
    f.write(']\n')
    f.write('PROMPT = "Explain the attention mechanism in transformers in detail."\n')
    f.write('MAX_NEW_TOKENS = 100\n')
    f.write('RUNS = 5\n')
    f.write('CSV_PATH = "/content/modelscope/results/benchmark_results.csv"\n')
    f.write('def get_already_done():\n')
    f.write('    if not os.path.exists(CSV_PATH): return []\n')
    f.write('    with open(CSV_PATH) as f: return [row["variant"] for row in __import__("csv").DictReader(f)]\n')
    f.write('def write_row(row):\n')
    f.write('    exists = os.path.exists(CSV_PATH)\n')
    f.write('    with open(CSV_PATH, "a", newline="") as f:\n')
    f.write('        w = __import__("csv").DictWriter(f, fieldnames=row.keys())\n')
    f.write('        if not exists: w.writeheader()\n')
    f.write('        w.writerow(row)\n')
    f.write('def measure_variant(v):\n')
    f.write('    print("=" * 50)\n')
    f.write('    print("Loading " + v["variant"] + "...")\n')
    f.write('    torch.cuda.reset_peak_memory_stats()\n')
    f.write('    mem_before = torch.cuda.memory_allocated() / (1024**2)\n')
    f.write('    t_load = time.time()\n')
    f.write('    kwargs = {"device_map": "auto", "torch_dtype": torch.float16}\n')
    f.write('    if v["quant_config"]:\n')
    f.write('        kwargs["quantization_config"] = v["quant_config"]\n')
    f.write('        del kwargs["torch_dtype"]\n')
    f.write('    model = AutoModelForCausalLM.from_pretrained(v["model_id"], **kwargs)\n')
    f.write('    tokenizer = AutoTokenizer.from_pretrained(v["model_id"])\n')
    f.write('    load_time = time.time() - t_load\n')
    f.write('    memory_mb = round((torch.cuda.max_memory_allocated() / (1024**2)) - mem_before, 1)\n')
    f.write('    print("Loaded in " + str(round(load_time,1)) + "s | Memory: " + str(memory_mb) + "MB")\n')
    f.write('    inputs = tokenizer(PROMPT, return_tensors="pt").to("cuda")\n')
    f.write('    ttft_list, tps_list = [], []\n')
    f.write('    for _ in tqdm(range(RUNS)):\n')
    f.write('        torch.cuda.synchronize()\n')
    f.write('        t_start = time.perf_counter()\n')
    f.write('        with torch.no_grad():\n')
    f.write('            outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.eos_token_id)\n')
    f.write('        torch.cuda.synchronize()\n')
    f.write('        t_end = time.perf_counter()\n')
    f.write('        n_tokens = outputs.shape[1] - inputs["input_ids"].shape[1]\n')
    f.write('        total_time = t_end - t_start\n')
    f.write('        ttft_list.append((total_time / max(n_tokens, 1)) * 1000)\n')
    f.write('        tps_list.append(n_tokens / total_time)\n')
    f.write('    result = {"variant": v["variant"], "model_id": v["model_id"], "memory_mb": memory_mb, "load_time_s": round(load_time,2), "ttft_p50_ms": round(float(np.percentile(ttft_list,50)),2), "ttft_p95_ms": round(float(np.percentile(ttft_list,95)),2), "tokens_per_sec": round(float(np.mean(tps_list)),2)}\n')
    f.write('    print("Result: " + str(result))\n')
    f.write('    del model\n')
    f.write('    gc.collect()\n')
    f.write('    torch.cuda.empty_cache()\n')
    f.write('    return result\n')
    f.write('already_done = get_already_done()\n')
    f.write('print("Already completed: " + str(already_done))\n')
    f.write('for v in tqdm(VARIANTS, desc="Overall Progress"):\n')
    f.write('    if v["variant"] in already_done:\n')
    f.write('        print("Skipping " + v["variant"])\n')
    f.write('        continue\n')
    f.write('    try:\n')
    f.write('        row = measure_variant(v)\n')
    f.write('        write_row(row)\n')
    f.write('        print("Saved: " + v["variant"])\n')
    f.write('    except Exception as e:\n')
    f.write('        print("ERROR on " + v["variant"] + ": " + str(e))\n')
    f.write('        continue\n')
    f.write('print("All benchmarks complete!")\n')
print('benchmark runner.py written')


In [ ]:
# CELL 4: Execute benchmark runner - 60 to 90 minutes, do not close tab
import subprocess, sys, os
env = os.environ.copy()
env['PYTHONPATH'] = '/content/modelscope'
process = subprocess.Popen([sys.executable, 'benchmarks/runner.py'], cwd='/content/modelscope', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
print('Benchmark finished with exit code: ' + str(process.returncode))


In [ ]:
# CELL 5: Load and display benchmark results with key insights
import pandas as pd
df_bench = pd.read_csv('/content/modelscope/results/benchmark_results.csv')
print('Completed variants: ' + str(len(df_bench)) + ' / 8')
print(df_bench[['variant','memory_mb','tokens_per_sec','ttft_p50_ms','ttft_p95_ms']].to_string(index=False))
try:
    fp16_mem = df_bench[df_bench.variant=='llama-fp16']['memory_mb'].values[0]
    int4_mem = df_bench[df_bench.variant=='llama-int4']['memory_mb'].values[0]
    reduction = round((1 - int4_mem/fp16_mem)*100, 1)
    print('Key Finding: llama-int4 reduces GPU memory ' + str(reduction) + '% vs FP16 (' + str(fp16_mem) + 'MB to ' + str(int4_mem) + 'MB)')
except: pass


## Section 3: Quality Evaluation
Running three complementary quality metrics: MMLU accuracy (factual correctness), output consistency (ROUGE-L variance), and WikiText-2 perplexity (the standard metric used in quantization research papers like GPTQ and AWQ).

In [ ]:
# CELL 6: Write quality eval harness with MMLU, consistency, and perplexity
import os
with open('/content/modelscope/eval/harness.py', 'w') as f:
    f.write('import os, gc, csv, math, torch, numpy as np\n')
    f.write('from tqdm import tqdm\n')
    f.write('from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n')
    f.write('from datasets import load_dataset\n')
    f.write('from rouge_score import rouge_scorer\n')
    f.write('os.makedirs("/content/modelscope/results", exist_ok=True)\n')
    f.write('VARIANTS = [\n')
    f.write('    {"variant": "llama-fp16", "model_id": "meta-llama/Llama-3.2-3B-Instruct", "quant_config": None},\n')
    f.write('    {"variant": "llama-int8", "model_id": "meta-llama/Llama-3.2-3B-Instruct", "quant_config": BitsAndBytesConfig(load_in_8bit=True)},\n')
    f.write('    {"variant": "llama-int4", "model_id": "meta-llama/Llama-3.2-3B-Instruct", "quant_config": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)},\n')
    f.write('    {"variant": "llama-int4-nf4", "model_id": "meta-llama/Llama-3.2-3B-Instruct", "quant_config": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)},\n')
    f.write('    {"variant": "gemma2-fp16", "model_id": "google/gemma-2-2b-it", "quant_config": None},\n')
    f.write('    {"variant": "gemma2-int8", "model_id": "google/gemma-2-2b-it", "quant_config": BitsAndBytesConfig(load_in_8bit=True)},\n')
    f.write('    {"variant": "gemma2-int4", "model_id": "google/gemma-2-2b-it", "quant_config": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)},\n')
    f.write('    {"variant": "gemma2-int4-nf4", "model_id": "google/gemma-2-2b-it", "quant_config": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)},\n')
    f.write(']\n')
    f.write('CSV_PATH = "/content/modelscope/results/quality_results.csv"\n')
    f.write('SUBJECTS = ["abstract_algebra", "anatomy", "astronomy", "computer_security"]\n')
    f.write('CONSISTENCY_PROMPT = "What are the key differences between supervised and unsupervised learning?"\n')
    f.write('def get_already_done():\n')
    f.write('    if not os.path.exists(CSV_PATH): return []\n')
    f.write('    with open(CSV_PATH) as f: return [row["variant"] for row in __import__("csv").DictReader(f)]\n')
    f.write('def write_row(row):\n')
    f.write('    exists = os.path.exists(CSV_PATH)\n')
    f.write('    with open(CSV_PATH, "a", newline="") as f:\n')
    f.write('        w = __import__("csv").DictWriter(f, fieldnames=row.keys())\n')
    f.write('        if not exists: w.writeheader()\n')
    f.write('        w.writerow(row)\n')
    f.write('def load_model(v):\n')
    f.write('    kwargs = {"device_map": "auto", "torch_dtype": torch.float16}\n')
    f.write('    if v["quant_config"]: kwargs["quantization_config"] = v["quant_config"]; del kwargs["torch_dtype"]\n')
    f.write('    model = AutoModelForCausalLM.from_pretrained(v["model_id"], **kwargs)\n')
    f.write('    tokenizer = AutoTokenizer.from_pretrained(v["model_id"])\n')
    f.write('    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token\n')
    f.write('    return model, tokenizer\n')
    f.write('def load_mmlu():\n')
    f.write('    questions = []\n')
    f.write('    for subject in SUBJECTS:\n')
    f.write('        ds = load_dataset("cais/mmlu", subject, split="test")\n')
    f.write('        for item in ds.select(range(min(25, len(ds)))):\n')
    f.write('            questions.append({"question": item["question"], "choices": item["choices"], "answer": item["answer"]})\n')
    f.write('    print("Loaded " + str(len(questions)) + " MMLU questions")\n')
    f.write('    return questions\n')
    f.write('def score_mmlu(model, tokenizer, questions):\n')
    f.write('    correct = 0\n')
    f.write('    for q in tqdm(questions, desc="MMLU"):\n')
    f.write('        messages = [{"role": "user", "content": "Question: " + q["question"] + "\\nA: " + q["choices"][0] + "\\nB: " + q["choices"][1] + "\\nC: " + q["choices"][2] + "\\nD: " + q["choices"][3] + "\\nAnswer with A, B, C, or D only:"}]\n')
    f.write('        try:\n')
    f.write('            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n')
    f.write('        except Exception:\n')
    f.write('            prompt = messages[0]["content"]\n')
    f.write('        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")\n')
    f.write('        with torch.no_grad():\n')
    f.write('            outputs = model.generate(**inputs, max_new_tokens=5, do_sample=False, pad_token_id=tokenizer.eos_token_id)\n')
    f.write('        decoded = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()\n')
    f.write('        pred = decoded[0].upper() if decoded else "X"\n')
    f.write('        if pred == ["A","B","C","D"][q["answer"]]: correct += 1\n')
    f.write('    return round(correct / len(questions), 4)\n')
    f.write('def score_consistency(model, tokenizer):\n')
    f.write('    messages = [{"role": "user", "content": CONSISTENCY_PROMPT}]\n')
    f.write('    try:\n')
    f.write('        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n')
    f.write('    except Exception:\n')
    f.write('        prompt = CONSISTENCY_PROMPT\n')
    f.write('    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")\n')
    f.write('    outputs_list = []\n')
    f.write('    for _ in tqdm(range(5), desc="Consistency"):\n')
    f.write('        with torch.no_grad():\n')
    f.write('            out = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7, pad_token_id=tokenizer.eos_token_id)\n')
    f.write('        outputs_list.append(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))\n')
    f.write('    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)\n')
    f.write('    scores = [scorer.score(outputs_list[i], outputs_list[j])["rougeL"].fmeasure for i in range(len(outputs_list)) for j in range(i+1, len(outputs_list))]\n')
    f.write('    return round(float(np.mean(scores)), 4)\n')
    f.write('def measure_perplexity(model, tokenizer):\n')
    f.write('    try:\n')
    f.write('        ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")\n')
    f.write('        texts = [t for t in ds["text"] if len(t.strip()) > 50][:80]\n')
    f.write('        total_loss, total_tokens = 0, 0\n')
    f.write('        for text in tqdm(texts, desc="Perplexity"):\n')
    f.write('            enc = tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to("cuda")\n')
    f.write('            if enc.input_ids.shape[1] < 2: continue\n')
    f.write('            with torch.no_grad():\n')
    f.write('                loss = model(**enc, labels=enc.input_ids).loss\n')
    f.write('            total_loss += loss.item() * (enc.input_ids.shape[1] - 1)\n')
    f.write('            total_tokens += enc.input_ids.shape[1] - 1\n')
    f.write('        return round(math.exp(total_loss / total_tokens), 2)\n')
    f.write('    except Exception as e:\n')
    f.write('        print("Perplexity error: " + str(e))\n')
    f.write('        return None\n')
    f.write('questions = load_mmlu()\n')
    f.write('already_done = get_already_done()\n')
    f.write('print("Already completed: " + str(already_done))\n')
    f.write('for v in tqdm(VARIANTS, desc="Overall Eval Progress"):\n')
    f.write('    if v["variant"] in already_done:\n')
    f.write('        print("Skipping " + v["variant"])\n')
    f.write('        continue\n')
    f.write('    try:\n')
    f.write('        print("Evaluating " + v["variant"] + "...")\n')
    f.write('        model, tokenizer = load_model(v)\n')
    f.write('        mmlu_acc = score_mmlu(model, tokenizer, questions)\n')
    f.write('        consistency = score_consistency(model, tokenizer)\n')
    f.write('        perplexity = measure_perplexity(model, tokenizer)\n')
    f.write('        write_row({"variant": v["variant"], "model_id": v["model_id"], "mmlu_accuracy": mmlu_acc, "consistency_score": consistency, "perplexity": perplexity})\n')
    f.write('        print("Saved: " + v["variant"] + " MMLU=" + str(mmlu_acc) + " PPL=" + str(perplexity))\n')
    f.write('        del model\n')
    f.write('        gc.collect()\n')
    f.write('        torch.cuda.empty_cache()\n')
    f.write('    except Exception as e:\n')
    f.write('        print("ERROR on " + v["variant"] + ": " + str(e))\n')
    f.write('        continue\n')
    f.write('print("All evals complete!")\n')
print('eval harness.py written')


In [ ]:
# CELL 7: Execute quality eval harness - 60 to 90 minutes, do not close tab
import subprocess, sys, os
env = os.environ.copy()
env['PYTHONPATH'] = '/content/modelscope'
process = subprocess.Popen([sys.executable, 'eval/harness.py'], cwd='/content/modelscope', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
print('Eval finished with exit code: ' + str(process.returncode))


In [ ]:
# CELL 8: Load and display eval results with key insights
import pandas as pd
df_qual = pd.read_csv('/content/modelscope/results/quality_results.csv')
print('Evaluated variants: ' + str(len(df_qual)))
print(df_qual[['variant','mmlu_accuracy','consistency_score','perplexity']].to_string(index=False))
try:
    fp16_ppl = df_qual[df_qual.variant=='llama-fp16']['perplexity'].values[0]
    int4_ppl = df_qual[df_qual.variant=='llama-int4']['perplexity'].values[0]
    fp16_mmlu = df_qual[df_qual.variant=='llama-fp16']['mmlu_accuracy'].values[0]
    int4_mmlu = df_qual[df_qual.variant=='llama-int4']['mmlu_accuracy'].values[0]
    print('Key Finding 1: llama-int4 perplexity delta vs FP16: ' + str(round(int4_ppl - fp16_ppl, 2)) + ' points')
    print('Key Finding 2: llama-int4 MMLU within ' + str(round(abs(int4_mmlu-fp16_mmlu)*100,1)) + '% of FP16')
except: pass


## Section 4: Pareto Analysis and Visualization
Identifying non-dominated configurations across three axes simultaneously: throughput, memory efficiency, and quality score.

In [ ]:
# CELL 9: Merge results, compute Pareto frontier, generate 4 publication-quality plots
import pandas as pd, numpy as np, matplotlib.pyplot as plt, json, os
os.makedirs('/content/modelscope/results/plots', exist_ok=True)
bench = pd.read_csv('/content/modelscope/results/benchmark_results.csv')
qual = pd.read_csv('/content/modelscope/results/quality_results.csv')
df = pd.merge(bench, qual, on='variant')
print('Merged variants: ' + str(len(df)))
def is_pareto(df):
    pareto = []
    for i, row in df.iterrows():
        dominated = any(
            df.loc[j,'tokens_per_sec'] >= row['tokens_per_sec'] and
            df.loc[j,'memory_mb'] <= row['memory_mb'] and
            df.loc[j,'mmlu_accuracy'] >= row['mmlu_accuracy'] and
            (df.loc[j,'tokens_per_sec'] > row['tokens_per_sec'] or
             df.loc[j,'memory_mb'] < row['memory_mb'] or
             df.loc[j,'mmlu_accuracy'] > row['mmlu_accuracy'])
            for j in df.index if j != i
        )
        pareto.append(not dominated)
    df['is_pareto'] = pareto
    return df
df = is_pareto(df)
colors = ['#1f77b4' if 'llama' in v else '#ff7f0e' for v in df['variant']]
sizes = [280 if p else 100 for p in df['is_pareto']]
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
fig.suptitle('ModelScope: Llama-3.2-3B vs Gemma-2-2B Inference Optimization\n8 quantization variants benchmarked on CUDA T4 GPU', fontsize=14, fontweight='bold')
ax = axes[0,0]
ax.scatter(df['memory_mb'], df['tokens_per_sec'], c=colors, s=sizes, alpha=0.85, edgecolors='black', linewidth=0.5)
for _, r in df.iterrows():
    ax.annotate(r['variant'], (r['memory_mb'], r['tokens_per_sec']), textcoords='offset points', xytext=(6,4), fontsize=8)
ax.set_xlabel('GPU Memory (MB)')
ax.set_ylabel('Tokens/sec')
ax.set_title('Memory vs Throughput\n(blue=llama, orange=gemma, large=Pareto optimal)')
ax.grid(True, alpha=0.3)
ax = axes[0,1]
ax.scatter(df['tokens_per_sec'], df['mmlu_accuracy'], c=colors, s=sizes, alpha=0.85, edgecolors='black', linewidth=0.5)
for _, r in df.iterrows():
    ax.annotate(r['variant'], (r['tokens_per_sec'], r['mmlu_accuracy']), textcoords='offset points', xytext=(6,4), fontsize=8)
ax.set_xlabel('Tokens/sec')
ax.set_ylabel('MMLU Accuracy')
ax.set_title('Quality vs Speed\n(higher-right is better)')
ax.grid(True, alpha=0.3)
ax = axes[1,0]
sc = ax.scatter(df['memory_mb'], df['tokens_per_sec'], c=df['mmlu_accuracy'], s=df['mmlu_accuracy']*800+50, cmap='YlOrRd', alpha=0.85, edgecolors='black', linewidth=0.5)
for _, r in df.iterrows():
    ax.annotate(r['variant'], (r['memory_mb'], r['tokens_per_sec']), textcoords='offset points', xytext=(6,4), fontsize=8)
plt.colorbar(sc, ax=ax, label='MMLU Accuracy')
ax.set_xlabel('GPU Memory (MB)')
ax.set_ylabel('Tokens/sec')
ax.set_title('Pareto Frontier\n(bubble size and color = quality)')
ax.grid(True, alpha=0.3)
ax = axes[1,1]
df_ppl = df.dropna(subset=['perplexity'])
if len(df_ppl) > 0:
    bar_colors = ['#1f77b4' if 'llama' in v else '#ff7f0e' for v in df_ppl['variant']]
    ax.bar(df_ppl['variant'], df_ppl['perplexity'], color=bar_colors, alpha=0.8, edgecolor='black')
    if 'llama-fp16' in df_ppl['variant'].values:
        baseline_llama = df_ppl[df_ppl['variant']=='llama-fp16']['perplexity'].values[0]
        ax.axhline(y=baseline_llama, color='#1f77b4', linestyle='--', alpha=0.7, label='Llama FP16 baseline')
    if 'gemma2-fp16' in df_ppl['variant'].values:
        baseline_gemma = df_ppl[df_ppl['variant']=='gemma2-fp16']['perplexity'].values[0]
        ax.axhline(y=baseline_gemma, color='#ff7f0e', linestyle='--', alpha=0.7, label='Gemma FP16 baseline')
    ax.legend(fontsize=8)
ax.set_ylabel('Perplexity (lower is better)')
ax.set_title('Perplexity Degradation by Quantization\n(vs FP16 baseline)')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('/content/modelscope/results/plots/final_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plots saved.')
df_norm = df.copy()
for col, maximize in [('tokens_per_sec', True), ('memory_mb', False), ('mmlu_accuracy', True)]:
    mn, mx = df[col].min(), df[col].max()
    df_norm[col+'_norm'] = (df[col]-mn)/(mx-mn) if mx > mn else 0.5
    if not maximize: df_norm[col+'_norm'] = 1 - df_norm[col+'_norm']
df_norm['balanced_score'] = df_norm[['tokens_per_sec_norm','memory_mb_norm','mmlu_accuracy_norm']].mean(axis=1)
rec = {
    'latency_priority': {**df.loc[df['tokens_per_sec'].idxmax(), ['variant','memory_mb','tokens_per_sec','mmlu_accuracy']].to_dict(), 'rationale': 'Highest throughput config for latency-sensitive production workloads'},
    'memory_priority': {**df.loc[df['memory_mb'].idxmin(), ['variant','memory_mb','tokens_per_sec','mmlu_accuracy']].to_dict(), 'rationale': 'Lowest GPU memory footprint for memory-constrained deployment environments'},
    'balanced': {**df_norm.loc[df_norm['balanced_score'].idxmax(), ['variant','memory_mb','tokens_per_sec','mmlu_accuracy']].to_dict(), 'rationale': 'Best normalized score across speed, memory, and quality for general use'},
}
with open('/content/modelscope/results/recommendation.json','w') as f:
    json.dump(rec, f, indent=2)
print('Recommendation saved.')
print(json.dumps(rec, indent=2))


## Section 5: Deployment Recommendations
Translating benchmark and eval results into actionable deployment decisions for three common production scenarios.

In [ ]:
# CELL 10: Print structured deployment recommendations
import json
with open('/content/modelscope/results/recommendation.json') as f:
    rec = json.load(f)
print('DEPLOYMENT RECOMMENDATIONS')
print('='*60)
for scenario, config in rec.items():
    if isinstance(config, dict):
        print('')
        print('Scenario: ' + scenario.upper().replace('_', ' '))
        print('  Config:     ' + str(config.get('variant', 'N/A')))
        print('  Memory:     ' + str(config.get('memory_mb', 'N/A')) + ' MB')
        print('  Throughput: ' + str(config.get('tokens_per_sec', 'N/A')) + ' tokens/sec')
        print('  MMLU:       ' + str(config.get('mmlu_accuracy', 'N/A')))
        print('  Rationale:  ' + str(config.get('rationale', 'N/A')))


## Section 6: Save and Export
Persisting all results to Google Drive and GitHub for reproducibility and portfolio access.

In [ ]:
# CELL 11: Save all results to Google Drive
import shutil, os
shutil.copytree('/content/modelscope/results', '/content/drive/MyDrive/modelscope_results', dirs_exist_ok=True)
print('Results saved to Google Drive.')
for root, dirs, files in os.walk('/content/modelscope/results'):
    for file in files:
        path = os.path.join(root, file)
        print('  ' + path + ' (' + str(os.path.getsize(path)) + ' bytes)')


In [ ]:
# CELL 12: Push all results and code to GitHub
import subprocess, os
token = input('GitHub personal access token: ')
def run(cmd):
    r = subprocess.run(cmd, shell=True, cwd='/content/modelscope', capture_output=True, text=True)
    if r.stdout: print(r.stdout)
    if r.stderr: print(r.stderr)
run("git config user.email 'anuvthot@gmail.com'")
run("git config user.name 'ANUVIK2401'")
run('git remote set-url origin https://' + token + '@github.com/ANUVIK2401/modelscope.git')
run('git add results/ benchmarks/runner.py eval/harness.py notebooks/')
run('git commit -m "results: complete benchmark + eval + perplexity across 8 variants"')
run('git push origin main')
print('Push complete.')


In [ ]:
# CELL 13: Download key output files to local machine
from google.colab import files
files.download('/content/modelscope/results/plots/final_analysis.png')
files.download('/content/modelscope/results/benchmark_results.csv')
files.download('/content/modelscope/results/quality_results.csv')
files.download('/content/modelscope/results/recommendation.json')
print('Downloads initiated.')
